## Silver Transformation – Population

This notebook creates the Silver table capstone.silver.population.

**Dependency:**
Uses the base geography transformation from the _geography_base notebook.

Source:
Cleaned geography dataframe geo.

**Transformation logic:**
- Extracts population values from the Census dataset
- Safely converts population values to numeric format
- Filters invalid or missing values
- Keeps one record per location_id

**Output table:**
capstone.silver.population

**Purpose:**
Stores county-level population metrics linked to location_id for downstream analysis.

In [0]:
%run ./_geography_base

In [0]:
%python
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# geo is already prepared from _geography_base

population = (
  geo
  .withColumn("population_total", F.expr("try_cast(B01003_001E as long)"))
  .filter(F.col("population_total").isNotNull())
  .select(
    "location_id",
    "population_total",
    "ingestion_timestamp",
    "source_system"
  )
)

w = Window.partitionBy("location_id").orderBy(F.col("ingestion_timestamp").desc_nulls_last())

population = (
  population
  .withColumn("rn", F.row_number().over(w))
  .filter(F.col("rn") == 1)
  .drop("rn")
)

(population.write
 .mode("overwrite")
 .format("delta")
 .option("overwriteSchema", "true")
 .saveAsTable("capstone.silver.population")
)

print("Created: capstone.silver.population")
display(population.limit(20))